In [76]:
# !pip install optuna

In [77]:
# ! pip install kagglehub[pandas-datasets]

In [78]:
# ! pip install torchinfo

In [79]:
import pandas as pd
import numpy as np
import matplotlib.pyplot  as plt
import seaborn as sns
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
import optuna
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from imblearn.over_sampling import SMOTE

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)


In [80]:
# Check for GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cpu


In [81]:
# import kagglehub
# from kagglehub import KaggleDatasetAdapter

# # Set the path to the file you'd like to load
# # For this dataset, the main file is 'ai4i2020.csv'
# file_path = "ai4i2020.csv"

# # Load the latest version using the recommended dataset_load method
# df = kagglehub.dataset_load(
#   KaggleDatasetAdapter.PANDAS,
#   "stephanmatzka/predictive-maintenance-dataset-ai4i-2020",
#   file_path
# )

100%|██████████| 510k/510k [00:00<00:00, 9.89MB/s]


In [82]:
# # Save to your Drive
# df.to_csv('/content/drive/MyDrive/Colab Notebooks/Predictive Maintenance Dataset (AI4I 2020)/ai4i2020_saved.csv', index=False)
# print("✅ Saved to Google Drive!")

In [83]:
# from google.colab import drive
# # drive.mount('/content/drive')

In [84]:
df=pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Predictive Maintenance Dataset (AI4I 2020)/ai4i2020_saved.csv')
df.head()

,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


In [110]:
df.shape

(10000, 12)

In [111]:
10000/32


312.5

In [85]:
# Drop non-predictive columns
df = df.drop(columns=['UDI', 'Product ID'])

In [86]:
pd.DataFrame({"Missing value":df.isna().sum(),"Datatype":df.dtypes})

,Missing value,Datatype
Type,0,object
Air temperature [K],0,float64
Process temperature [K],0,float64
Rotational speed [rpm],0,int64
Torque [Nm],0,float64
Tool wear [min],0,int64
Machine failure,0,int64
TWF,0,int64
HDF,0,int64
PWF,0,int64


In [87]:
X = df.drop('Machine failure', axis=1)
X.head()

,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],TWF,HDF,PWF,OSF,RNF
0,M,298.1,308.6,1551,42.8,0,0,0,0,0,0
1,L,298.2,308.7,1408,46.3,3,0,0,0,0,0
2,L,298.1,308.5,1498,49.4,5,0,0,0,0,0
3,L,298.2,308.6,1433,39.5,7,0,0,0,0,0
4,L,298.2,308.7,1408,40.0,9,0,0,0,0,0


In [88]:
y=df[["Machine failure"]]
y.head()

,Machine failure
0,0
1,0
2,0
3,0
4,0


In [89]:
df["Machine failure"].value_counts()

,count
Machine failure,
0,9661
1,339


### This is highly imbalanced data for this we should balance the data using SMOTE

In [90]:
# Stratified split to preserve failure ratio
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [91]:
# Encode categorical feature
le = LabelEncoder()
X_train['Type'] = le.fit_transform(X_train['Type'])
X_test['Type'] = le.transform(X_test['Type'])

In [92]:
# Apply SMOTE to training data only
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)


In [93]:
# Scale features
scaler = StandardScaler()
X_train_resampled = scaler.fit_transform(X_train_resampled)
X_test = scaler.transform(X_test)

In [94]:
y_train_resampled.value_counts()

,count
Machine failure,
0,7729
1,7729


In [95]:
# Custom Dataset and data loader
class CustomDataset(Dataset):
    def __init__(self,features,labels):
        self.features = torch.tensor(features.values if hasattr(features, 'values') else features, dtype=torch.float32)
        self.labels = torch.tensor(labels.values.ravel() if hasattr(labels, 'values') else labels.ravel(), dtype=torch.long)
    def __len__(self):
        return len(self.features)
    def __getitem__(self, index):
        return self.features[index],self.labels[index]

In [96]:
train_dataset = CustomDataset(X_train_resampled, y_train_resampled)

In [97]:
test_dataset = CustomDataset(X_test, y_test)

In [98]:
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [99]:
len(train_loader)

484

In [101]:
class PredictiveMaintenanceANN(nn.Module):
    def __init__(self,num_features):
        super().__init__()
        self.model=nn.Sequential(
            nn.Linear(num_features, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1)
        )
    def forward(self,x):
        return self.model(x)

In [102]:
# set learning rate and epochs
epochs = 100
learning_rate = 0.001

In [103]:
# instatiate the model
model=PredictiveMaintenanceANN(X_train_resampled.shape[1])
model

PredictiveMaintenanceANN(
  (model): Sequential(
    (0): Linear(in_features=11, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=128, out_features=64, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.3, inplace=False)
    (6): Linear(in_features=64, out_features=1, bias=True)
  )
)

In [104]:
from torchinfo import summary

summary(model, input_size=(1, 11))

Layer (type:depth-idx)                   Output Shape              Param #
PredictiveMaintenanceANN                 [1, 1]                    --
├─Sequential: 1-1                        [1, 1]                    --
│    └─Linear: 2-1                       [1, 128]                  1,536
│    └─ReLU: 2-2                         [1, 128]                  --
│    └─Dropout: 2-3                      [1, 128]                  --
│    └─Linear: 2-4                       [1, 64]                   8,256
│    └─ReLU: 2-5                         [1, 64]                   --
│    └─Dropout: 2-6                      [1, 64]                   --
│    └─Linear: 2-7                       [1, 1]                    65
Total params: 9,857
Trainable params: 9,857
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 0.01
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.04
Estimated Total Size (MB): 0.04

In [106]:
# Loss Function
criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([32.0]))

# optimizer
optimizer = optim.AdamW(model.parameters(), lr=learning_rate)


In [ ]:
# training loop

for epoch in range(epochs):

    total_epoch_loss = 0

    for batch_features,batch_labels in train_loader:
        
        # forward pass
        outputs = model(batch_features).view(-1)

        # calculate loss
        loss = criterion(outputs, batch_labels.float())

        # back pass
        optimizer.zero_grad()
        
        loss.backward()

        # update grads
        optimizer.step()

        total_epoch_loss = total_epoch_loss + loss.item()
    avg_loss = total_epoch_loss/len(train_loader)
    print(f'Epoch: {epoch + 1} , Loss: {avg_loss}')

Epoch: 1 , Loss: 1.2724304534187
Epoch: 2 , Loss: 0.7368758810022153
Epoch: 3 , Loss: 0.6816732882991556
Epoch: 4 , Loss: 0.6592557281255722
Epoch: 5 , Loss: 0.6122511437178151
Epoch: 6 , Loss: 0.5935514530490252
Epoch: 7 , Loss: 0.5688801669815847
Epoch: 8 , Loss: 0.5380584340511767
Epoch: 9 , Loss: 0.5568956575502665
Epoch: 10 , Loss: 0.5254519730121185
Epoch: 11 , Loss: 0.5044731555356498
Epoch: 12 , Loss: 0.5065210487766668
Epoch: 13 , Loss: 0.4882553754612988
Epoch: 14 , Loss: 0.4791035000560156
Epoch: 15 , Loss: 0.482170099765832
Epoch: 16 , Loss: 0.4735017538778792
Epoch: 17 , Loss: 0.4951665732424614
Epoch: 18 , Loss: 0.4800071114225713
Epoch: 19 , Loss: 0.4511533795984862
Epoch: 20 , Loss: 0.4444649140004161
Epoch: 21 , Loss: 0.450311076003781
Epoch: 22 , Loss: 0.45222269752285077
Epoch: 23 , Loss: 0.44048212113028223
Epoch: 24 , Loss: 0.4312954910045628
Epoch: 25 , Loss: 0.4167324042424929
Epoch: 26 , Loss: 0.43068854905634135
Epoch: 27 , Loss: 0.414171661588956
Epoch: 28 , L

In [108]:
# set model to eval mode
model.eval()

PredictiveMaintenanceANN(
  (model): Sequential(
    (0): Linear(in_features=11, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=128, out_features=64, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.3, inplace=False)
    (6): Linear(in_features=64, out_features=1, bias=True)
  )
)

In [113]:
# evaluation code
preds, labels = [], []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        logits = model(X_batch.to(device)).squeeze()      # Forward pass
        probs = torch.sigmoid(logits).cpu().numpy()       # Convert logits → probabilities
        preds.extend(probs)                               # Store for later
        labels.extend(y_batch.numpy())

preds = np.array(preds)
labels = np.array(labels)
y_pred = (np.array(preds) >= 0.5).astype(int)             # Threshold to 0/1 predictions

print("\n📊 Test Metrics:")
print(f"Accuracy:  {accuracy_score(labels, y_pred)}")
print(f"Precision: {precision_score(labels, y_pred)}")
print(f"Recall:    {recall_score(labels, y_pred)}")
print(f"F1-Score:  {f1_score(labels, y_pred)}")
print("\n📋 Confusion Matrix:")
cm = confusion_matrix(labels, y_pred)
# sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['No Failure', 'Failure'], yticklabels=['No Failure', 'Failure'])
# plt.xlabel('Predicted'); plt.ylabel('Actual'); plt.title('Test Confusion Matrix')
# plt.show()
cm



📊 Test Metrics:
Accuracy:  0.848
Precision: 0.1827956989247312
Recall:    1.0
F1-Score:  0.3090909090909091

📋 Confusion Matrix:


array([[1628,  304],
       [   0,   68]])